# Netflix-Datenbereinigungsprojekt (pandas + NumPy)

**Ablauf:** Geschäftsproblem → Laden → Untersuchung → Kopieren → Standardisierung von Spalten/Text → Duplikate → Fehlerhafte Werte → Fehlende Werte → Datentyp-Konvertierung → Feature-Engineering → Konsistenz/Ausreißer → Validierung → Speichern

## 0. Das Geschäftsproblem verstehen
Jede Zeile entspricht einem Film oder einer Serie auf Netflix. Ziel: eine zuverlässige Tabelle, die für Analysen und Dashboards bereit ist.

**Goldene Regeln:** Rohdaten nicht verändern · erst messen, dann ändern · keine Daten erfinden · jeden Schritt überprüfen.

## 1. Bibliotheken
`pandas` für Tabellenoperationen, `numpy` für bedingte und numerische Operationen.

In [6]:
import pandas as pd
import numpy as np

In [8]:
from pathlib import Path
import pandas as pd

# Lokale Datei verwenden, sonst direkt von GitHub laden
LOCAL_FILE = Path("netflix_titles.csv")
URL = (
    "https://raw.githubusercontent.com/thepycoach/python-for-data-science/main/"
    "11.Project%203%20-%20Data%20Cleaning/netflix_titles.csv"
)

df_raw = pd.read_csv(LOCAL_FILE if LOCAL_FILE.exists() else URL)

In [10]:
df_raw.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [11]:
# Form des Datensatzes (Zeilen, Spalten)
print("Shape:", df_raw.shape)

# Datentypen und Anzahl nicht fehlender Werte
df_raw.info()

Shape: (8807, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [12]:
# Zufällige Stichprobe ansehen (random_state macht sie reproduzierbar)
df_raw.sample(5, random_state=42)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
4970,s4971,Movie,"Game Over, Man!",Kyle Newacheck,"Adam DeVine, Anders Holm, Blake Anderson, Utka...",United States,"March 23, 2018",2018,TV-MA,102 min,"Action & Adventure, Comedies",Three buddies with big dreams go from underach...
3362,s3363,Movie,Arsenio Hall: Smart & Classy,Brian Volk-Weiss,Arsenio Hall,United States,"October 29, 2019",2019,TV-MA,63 min,Stand-Up Comedy,"In his first stand-up special, Arsenio Hall di..."
5494,s5495,TV Show,Kazoops!,NaN,"Reece Pockney, Scott Langley, Alex Babic, Gemm...",Australia,"May 5, 2017",2017,TV-Y,3 Seasons,Kids' TV,Music meets imagination in this inventive anim...
1688,s1689,TV Show,We Are the Champions,NaN,NaN,United States,"November 17, 2020",2020,TV-MA,1 Season,"Docuseries, Reality TV","Explore an array of unique competitions, from ..."
1349,s1350,TV Show,"Pablo Escobar, el patrón del mal",NaN,"Andrés Parra, Angie Cepeda, Cecilia Navia, Vic...",Colombia,"February 3, 2021",2012,TV-MA,1 Season,"Crime TV Shows, International TV Shows, Spanis...",From his days as a petty thief to becoming hea...


In [14]:
# Statistische Kennzahlen für numerische Spalten
df_raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8807,8807,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8807,2,Movie,6131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8807,8807,Dick Johnson Is Dead,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,6173,4528,Rajiv Chilaka,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cast,7982,7692,David Attenborough,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,7976,748,United States,2818,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8797,1767,"January 1, 2020",109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8807.0,NaN,NaN,NaN,2014.180198,8.819312,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8803,17,TV-MA,3207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8804,220,1 Season,1793,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Datenqualitätsbericht
Anzahl/Prozentsatz fehlender Werte, Anzahl eindeutiger Werte und Duplikate.

In [15]:
# Qualitätsbericht pro Spalte erstellen
quality_report = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "unique": df_raw.nunique(),
}).sort_values("missing", ascending=False)
quality_report

,dtype,missing,missing_pct,unique
director,object,2634,29.91,4528
country,object,831,9.44,748
cast,object,825,9.37,7692
date_added,object,10,0.11,1767
rating,object,4,0.05,17
duration,object,3,0.03,220
show_id,object,0,0.00,8807
type,object,0,0.00,2
title,object,0,0.00,8807
release_year,int64,0,0.00,74


In [16]:
# Duplikate prüfen: komplette Zeilen, ID und fachlicher Schlüssel
print("Doppelte Zeilen:", df_raw.duplicated().sum())
print("Doppelte show_id:", df_raw["show_id"].duplicated().sum())
print("Doppelt (title, type, release_year):",
      df_raw.duplicated(subset=["title", "type", "release_year"]).sum())

Doppelte Zeilen: 0
Doppelte show_id: 0
Doppelt (title, type, release_year): 0


In [17]:
# Häufigkeiten kategorialer Spalten: hier fallen falsche Werte auf
print(df_raw["type"].value_counts(dropna=False), end="\n\n")
print(df_raw["rating"].value_counts(dropna=False))

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64


**Befund:** In der Spalte `rating` finden sich Einträge wie `74 min`, `84 min` und `66 min`. Dabei handelt es sich nicht um Altersfreigaben, sondern um Zeitangaben; sie sind in der falschen Spalte gelandet.

## 5. Arbeitskopie und Spaltennamen

In [20]:
# Rohdaten unverändert lassen und auf einer Kopie arbeiten
df = df_raw.copy()

# Spaltennamen vereinheitlichen (snake_case)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Verständlichere Namen vergeben
df = df.rename(columns={"listed_in": "genres", "cast": "cast_members"})
df.columns.tolist()

['show_id',
 'type',
 'title',
 'director',
 'cast_members',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'genres',
 'description']

## 6. Textbereinigung
Führende/nachgestellte Leerzeichen, doppelte Leerzeichen, unsichtbare Zeichen sowie überflüssige Kommas am Rand von Listen.

In [21]:
# Alle Textspalten ermitteln
text_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]

for col in text_cols:
    df[col] = (df[col]
               .str.replace("\u200b", "", regex=False)   # unsichtbare Zeichen entfernen
               .str.replace(r"\s+", " ", regex=True)      # mehrfache Leerzeichen reduzieren
               .str.strip())                              # Ränder bereinigen

# Überflüssige Kommas am Anfang/Ende von Listenspalten entfernen
list_cols = ["director", "cast_members", "country", "genres"]
for col in list_cols:
    df[col] = df[col].str.replace(r"^[,\s]+|[,\s]+$", "", regex=True)

# Leere Zeichenketten als fehlende Werte behandeln
df[text_cols] = df[text_cols].replace("", np.nan)

In [22]:
# Duplikate entfernen und Anzahl protokollieren
rows_before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="show_id", keep="first")
print("Entfernte Zeilen:", rows_before - len(df))

Entfernte Zeilen: 0


In [23]:
# Zeilen finden, in denen eine Dauer im Rating steht
mask_wrong = df["rating"].str.fullmatch(r"\d+ min", na=False) & df["duration"].isna()
display(df.loc[mask_wrong, ["title", "rating", "duration"]])

# Wert verschieben und falsches Rating löschen
df.loc[mask_wrong, "duration"] = df.loc[mask_wrong, "rating"]
df.loc[mask_wrong, "rating"] = np.nan

# NR (Not Rated) und UR (Unrated) bedeuten dasselbe
df["rating"] = df["rating"].replace({"UR": "NR"})

,title,rating,duration
5541,Louis C.K. 2017,74 min,NaN
5794,Louis C.K.: Hilarious,84 min,NaN
5813,Louis C.K.: Live at the Comedy Store,66 min,NaN


## 9. Fehlende Werte
| Spalte | Entscheidung | Grund |
|---|---|---|
| director, cast_members, country | `Unknown` | Hoher Anteil; Löschen würde zu Datenverlust führen |
| rating | `Unknown` | Sehr wenige Fälle; der tatsächliche Wert ist unbekannt |
| date_added | leer lassen (`NaT`) | Ein Datum lässt sich nicht einfach erfinden |

Das Auffüllen des Landes mit dem Modus (dem häufigsten Wert) würde dazu führen, dass Hunderte von Produktionen fälschlicherweise den USA zugeordnet würden. Um die Information über das Fehlen der Daten nicht zu verlieren, fügen wir zunächst eine Indikatorspalte (Flag-Spalte) hinzu.

In [24]:
fill_cols = ["director", "cast_members", "country", "rating"]

# Markieren, wo ursprünglich Werte gefehlt haben
for col in ["director", "cast_members", "country"]:
    df[f"{col}_missing"] = df[col].isna()

# Fehlende Texte mit "Unknown" ersetzen
df[fill_cols] = df[fill_cols].fillna("Unknown")
df.isna().sum()

show_id                  0
type                     0
title                    0
director                 0
cast_members             0
country                  0
date_added              10
release_year             0
rating                   0
duration                 0
genres                   0
description              0
director_missing         0
cast_members_missing     0
country_missing          0
dtype: int64

## 10. Datentyp-Konvertierungen
`date_added`: Text → Datum. `errors="coerce"` wandelt nicht konvertierbare Werte in `NaT` um; daher prüfen wir nach der Konvertierung, ob Daten verloren gegangen sind.

In [25]:
# Datum konvertieren (Format z. B. "September 25, 2021")
missing_before = df["date_added"].isna().sum()
df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce")

# Kontrolle: Es dürfen keine neuen fehlenden Werte entstanden sein
assert df["date_added"].isna().sum() == missing_before
print(df["date_added"].min(), "->", df["date_added"].max())

2008-01-01 00:00:00 -> 2021-09-25 00:00:00


In [26]:
# Zahl aus dem Text extrahieren
duration_value = pd.to_numeric(df["duration"].str.extract(r"(\d+)")[0], errors="coerce")

# Getrennte Spalten für Filme (Minuten) und Serien (Staffeln); Int16 erlaubt fehlende Werte
df["duration_min"] = duration_value.where(df["type"].eq("Movie")).astype("Int16")
df["season_count"] = duration_value.where(df["type"].eq("TV Show")).astype("Int16")
df[["type", "duration", "duration_min", "season_count"]].sample(6, random_state=1)

,type,duration,duration_min,season_count
2894,TV Show,1 Season,<NA>,1
2161,Movie,72 min,72,<NA>
6363,Movie,74 min,74,<NA>
2575,TV Show,1 Season,<NA>,1
2698,TV Show,2 Seasons,<NA>,2
7223,Movie,96 min,96,<NA>


In [27]:
# Datumsbestandteile
df["year_added"] = df["date_added"].dt.year.astype("Int16")
df["month_added"] = df["date_added"].dt.month.astype("Int8")

# Jahre zwischen Veröffentlichung und Aufnahme bei Netflix
df["years_to_netflix"] = df["year_added"] - df["release_year"]

# Hauptland und Hauptgenre (jeweils erster Eintrag der Liste)
df["main_country"] = df["country"].str.split(",").str[0].str.strip()
df["main_genre"] = df["genres"].str.split(",").str[0].str.strip()

# Zielgruppe aus dem Rating ableiten (np.select = mehrere Bedingungen)
conditions = [
    df["rating"].isin(["TV-Y", "TV-Y7", "TV-Y7-FV", "G", "TV-G"]),
    df["rating"].isin(["PG", "TV-PG"]),
    df["rating"].isin(["PG-13", "TV-14"]),
    df["rating"].isin(["R", "NC-17", "TV-MA"]),
]
choices = ["Kids", "Family", "Teens", "Adults"]
df["audience"] = np.select(conditions, choices, default="Unknown")

In [28]:
# Unlogisch: vor dem Erscheinungsjahr hinzugefügt
df["date_conflict"] = df["years_to_netflix"].lt(0).fillna(False).astype(bool)
print("Datumskonflikte:", df["date_conflict"].sum())
df.loc[df["date_conflict"], ["title", "type", "release_year", "date_added"]].head()

Datumskonflikte: 14


,title,type,release_year,date_added
1551,Hilda,TV Show,2021,2020-12-14
1696,Polly Pocket,TV Show,2021,2020-11-15
2920,Love Is Blind,TV Show,2021,2020-02-13
3168,Fuller House,TV Show,2020,2019-12-06
3287,Maradona in Mexico,TV Show,2020,2019-11-13


In [29]:
# Ausreißer bei der Filmdauer mit der IQR-Methode
movie_minutes = df["duration_min"].dropna().to_numpy(dtype=float)
q1, q3 = np.percentile(movie_minutes, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

df["duration_outlier"] = (
    (df["duration_min"].lt(lower) | df["duration_min"].gt(upper)).fillna(False).astype(bool)
)
print(f"Grenzen: {lower:.1f} - {upper:.1f} Minuten")
print("Ausreißer:", df["duration_outlier"].sum())
df.loc[df["duration_outlier"]].sort_values("duration_min")[["title", "duration_min"]].iloc[[0, 1, -2, -1]]

Grenzen: 46.5 - 154.5 Minuten
Ausreißer: 450


,title,duration_min
3777,Silent,3
2713,Sol Levante,5
717,Headspace: Unwind Your Mind,273
4253,Black Mirror: Bandersnatch,312


In [30]:
memory_before = df.memory_usage(deep=True).sum() / 1024**2

for col in ["type", "rating", "audience"]:
    df[col] = df[col].astype("category")

memory_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Speicher: {memory_before:.2f} MB -> {memory_after:.2f} MB")

Speicher: 9.29 MB -> 7.95 MB


In [32]:
def explode_column(data, col):
    """Zerlegt eine kommagetrennte Spalte in eine Zeile pro Wert."""
    result = data[["show_id", col]].assign(**{col: data[col].str.split(",")}).explode(col)
    result[col] = result[col].str.strip()
    return result.drop_duplicates().reset_index(drop=True)

df_countries = explode_column(df, "country")
df_genres = explode_column(df, "genres")
df_countries["country"].value_counts().head(10)

country
United States     3690
India             1046
Unknown            831
United Kingdom     806
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Name: count, dtype: int64

In [33]:
# Qualitätsregeln automatisch prüfen
assert len(df) == len(df_raw), "Unerwarteter Zeilenverlust"
assert df["show_id"].is_unique, "show_id ist nicht eindeutig"
assert df[fill_cols].notna().all().all(), "Fehlende Werte in gefüllten Spalten"
assert not df["rating"].astype(str).str.contains("min").any(), "Dauer steht im Rating"
assert df.loc[df["type"] == "Movie", "duration_min"].notna().all(), "Film ohne Dauer"
assert df.loc[df["type"] == "TV Show", "season_count"].notna().all(), "Serie ohne Staffelzahl"
assert df["release_year"].between(1900, df_raw["release_year"].max()).all(), "Ungültiges Jahr"
print("Alle Prüfungen bestanden")
df.info()

Alle Prüfungen bestanden
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   show_id               8807 non-null   object        
 1   type                  8807 non-null   category      
 2   title                 8807 non-null   object        
 3   director              8807 non-null   object        
 4   cast_members          8807 non-null   object        
 5   country               8807 non-null   object        
 6   date_added            8797 non-null   datetime64[ns]
 7   release_year          8807 non-null   int64         
 8   rating                8807 non-null   category      
 9   duration              8807 non-null   object        
 10  genres                8807 non-null   object        
 11  description           8807 non-null   object        
 12  director_missing      8807 non-null   bool         

In [34]:
# Ausgabeordner anlegen und bereinigte Daten speichern
output_dir = Path("data_clean")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "netflix_clean.csv", index=False)
df_countries.to_csv(output_dir / "netflix_countries.csv", index=False)
df_genres.to_csv(output_dir / "netflix_genres.csv", index=False)

# Kontrolle durch erneutes Einlesen
check = pd.read_csv(output_dir / "netflix_clean.csv", parse_dates=["date_added"])
assert check.shape == df.shape
print("Gespeichert:", check.shape)

Gespeichert: (8807, 25)
